<a href="https://colab.research.google.com/github/i1lmatic/simulador-saturacion-devops-unamad/blob/main/DSS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EDO Acoplada Formalización Matemático-Sistémica (semana 3)

Para modelar el comportamiento no lineal de la infraestructura de la UNAMAD durante las matrículas, traducimos el diagrama causal en un sistema de tres Ecuaciones Diferenciales Ordinarias (EDO) acopladas.

##  1. Definición Formal de Variables de Estado (Stocks)
1. **$Q(t)$ (Cola de Solicitudes en Memoria):** Representa el volumen físico de peticiones HTTP retenidas en el buffer de red esperando ser procesadas. [Unidad: Peticiones]
2. **$A(t)$ (Alertas Activas en el Buffer de Monitoreo):** Densidad de notificaciones críticas generadas por segundo que saturan las consolas de operaciones. [Unidad: Alertas]
3. **$F(t)$ (Fatiga Cognitiva Acumulada del Operador):** Nivel de desgaste psico-informático del equipo de ingenieros DevOps/SRE. [Unidad: Adimensional, escala de 0 a 1]

##  2. Tasas de Cambio (Flujos) y Ecuaciones Diferenciales

### A. Ecuación de la Cola de Solicitudes ($dQ/dt$)
$$\frac{dQ}{dt} = \lambda(t) - \mu \cdot (1 - F(t)) \cdot M_H(Q)$$
* **$\lambda(t)$:** Tasa estocástica de entrada de peticiones de los estudiantes.
* **$\mu$:** Capacidad nominal máxima de procesamiento del clúster de servidores.
* **$(1 - F(t))$:** Factor de degradación humana; a mayor fatiga del ingeniero, disminuye la velocidad para optimizar los parches en caliente.

### B. Ecuación de Alertas Activas ($dA/dt$)
$$\frac{dA/dt} = \alpha \cdot \left(\frac{Q(t)}{K_Q}\right) - \beta \cdot A(t)$$
* **$\alpha$:** Tasa de generación de alertas automatizadas disparadas por la saturación de la cola.
* **$\beta$:** Tasa de atenuación o despacho de alertas conforme se cierran los incidentes.

### C. Ecuación de Fatiga Cognitiva Acumulada ($dF/dt$)
$$\frac{dF}{dt} = \gamma \cdot A(t) \cdot (1 - F(t)) - \delta \cdot F(t)$$
* **$\gamma$:** Coeficiente de inducción de estrés por el bombardeo visual de alertas.
* **$\delta$:** Tasa de recuperación cognitiva (descanso o rotación del operador).

---

##  3. Multiplicador Logístico de la Restricción del Entorno Hardware ($M_H$)
En cumplimiento estricto con la restricción del entorno **(E)** del CATWOE, el hardware posee límites físicos insuperables (ancho de banda regional y memoria física asignada). Esto se modela mediante el **Multiplicador Logístico de Saturación**:

$$M_H(Q) = 1 - \left(\frac{Q(t)}{K_{max}}\right)^2$$

Donde **$K_{max}$** representa el techo asintótico de almacenamiento del buffer. A medida que la cola $Q(t)$ se aproxima a $K_{max}$, el multiplicador cae drásticamente a cero ($M_H \to 0$), provocando el colapso (denegación de servicio o *Crash* del software de matrícula) por rigidez estructural del hardware.

In [2]:
import numpy as np
from scipy.integrate import solve_ivp

def simulador_devops_unamad(t: float, y: list, params: dict) -> list:
    """
    Define el sistema de Ecuaciones Diferenciales Ordinarias (EDO)
    para la saturación de infraestructura y fatiga del operador SRE.
    """
    # Desempaquetar las Variables de Estado (Stocks)
    Q, A, F = y

    # Desempaquetar Parámetros de las Tasas (Rates)
    mu = params['mu']
    K_max = params['K_max']
    alpha = params['alpha']
    K_Q = params['K_Q']
    beta = params['beta']
    gamma = params['gamma']
    delta = params['delta']

    # 1. Tasa de entrada variable (Simulación de ráfaga de matrícula a los t=10 segundos)
    lambda_t = 150.0 if 10.0 <= t <= 40.0 else 15.0

    # 2. Multiplicador Logístico (Restricción del Entorno Hardware - E)
    # Se añade un clip de seguridad para evitar raíces o valores negativos inválidos en desbordamiento
    Q_seguro = max(0.0, min(Q, K_max))
    M_H = 1.0 - (Q_seguro / K_max)**2

    # 3. Flujos de Salida y Ecuaciones Diferenciales Acopladas
    # dQ/dt: Entrada - Procesamiento degradado por la fatiga y limitado por el Hardware
    dQ_dt = lambda_t - (mu * (1.0 - F) * M_H)

    # dA/dt: Alertas generadas por tamaño de cola - Alertas atendidas/vencidas
    dA_dt = alpha * (Q_seguro / K_Q) - (beta * A)

    # dF/dt: Estrés inducido por alertas activas limitado por el techo de fatiga (1.0) - Recuperación
    dF_dt = gamma * A * (1.0 - F) - (delta * F)

    return [dQ_dt, dA_dt, dF_dt]

# --- Configuración Inicial de Prueba del Laboratorio Virtual ---
parametros_sistema = {
    'mu': 80.0,      # Capacidad de procesamiento (peticiones/seg)
    'K_max': 1000.0,  # Capacidad máxima del buffer físico de memoria
    'alpha': 2.5,     # Tasa de generación de alertas
    'K_Q': 200.0,     # Umbral crítico de cola para disparar alertas
    'beta': 0.8,      # Atenuación de alertas
    'gamma': 0.05,    # Factor de desgaste por estrés
    'delta': 0.1      # Factor de recuperación humana
}

# Condiciones iniciales [Cola inicial, Alertas iniciales, Fatiga inicial]
condiciones_iniciales = [10.0, 0.0, 0.0]
tiempo_simulacion = (0.0, 60.0) # Segundos de simulación

print("¡Estructura de la Semana 3 inicializada correctamente!")

¡Estructura de la Semana 3 inicializada correctamente!
